# VATEX - kontrola poprawności implementacji potoku bazowego

Sprawdza, czy potok bazowy odtwarza wartości Recall@k publikowane dla VATEX. Potok bez segmentacji: każdy klip jest jednym fragmentem, aktywny jest wyłącznie sygnał sceniczny (OpenCLIP ViT-H/14). Korzysta z tych samych komponentów `src/`, co skrypt `scripts/vatex_check.py`.

**Wymaga:** `notebooks/prepare_data/vatex_02_test_acquisition.ipynb` - stamtąd biorą się klipy i pliki pośrednie w `data/interim/vatex`.

**Zapisuje:**

| plik | co zawiera |
|---|---|
| `data/annotations/vatex/vatex_check.jsonl` | zapytania kontrolne, dziesięć opisów na klip |
| `data/cache/vatex/` | embeddingi klipów i indeks FAISS, budowane raz i potem czytane z dysku |

Plik kontrolny powstaje tutaj, a nie w notatniku przygotowania danych: czyta go wyłącznie ten notatnik i jego odpowiednik `scripts/vatex_check.py`. Zapytania testowe eksperymentów to osobny plik i osobny zakres klipów, `notebooks/prepare_data/vatex_03_test_annotations.ipynb`.

In [ ]:
from src.utils.notebook import setup

setup()

from src.evaluation.metrics import evaluate_matrix, query_metrics
from src.features.openclip import ClipEncoder
from src.retrieval.pipeline import Pipeline
from src.retrieval.signals import SceneSignal
from src.utils import config as conf
from src.utils import vatex
from src.utils.queries import load_jsonl
from src.vatex_pipeline import build_collection, write_check_queries

cfg = conf.load('configs/vatex_base.yaml')
ks = tuple(cfg['evaluation']['ks'])
cfg['model']

## 1. Zapytania kontrolne

**Zapisuje:** `data/annotations/vatex/vatex_check.jsonl` - wszystkie dziesięć opisów każdego klipu jako osobne zapytania, na pełnym zbiorze `ok` (2560 klipów).

Zbiór celowo obejmuje wszystkie klipy `ok`, także te z przeciekiem treningowym Kinetics-400: liczy się porównywalność z wartościami publikowanymi, a nie czystość względem treningu. Nie ma tu podziału na dev i test - miałby sens tylko wtedy, gdyby cokolwiek było na tym pliku strojone. Identyfikatory zapytań są numerowane po kolei i tak zostaje, bo nic wpisywanego ręcznie nie jest po nich kluczowane.

In [ ]:
report       = vatex.load_report(conf.path(cfg['paths']['report']))
descriptions = vatex.load_descriptions(conf.path(cfg['paths']['descriptions']))

clips_dir = conf.path(cfg['paths']['clips'])
ok_vids   = vatex.ok_clips(report, clips_dir)      # 2560 -- candidate pool of the check

check   = write_check_queries(ok_vids, descriptions, cfg)
queries = load_jsonl(check)

print(f'ok clips:      {len(ok_vids)}  (candidate pool of the check)')
print(f'check queries: {len(queries)}  ({len(queries) // len(ok_vids)} descriptions/clip)')
print(f'saved to:      {check.relative_to(conf.ROOT)}')

# example record
queries[0].to_dict()

## 2. Indeks sceniczny FAISS

**Zapisuje:** `data/cache/vatex/` - embeddingi wszystkich klipów `ok` i indeks FAISS. Kolekcja budowana jest przy pierwszym uruchomieniu, a kolejne wczytują ją z dysku.

Enkoder OpenCLIP wczytywany jest raz. Pełne kodowanie około 2,5 tys. klipów trwa kilkanaście minut; jeśli w tle działa `scripts/vatex_check.py`, ten blok jedynie wczyta gotowy indeks.

In [ ]:
encoder    = ClipEncoder(cfg['model']['encoder'], cfg['model']['weights'])
collection = build_collection(ok_vids, encoder, cfg)
pipeline   = Pipeline(collection, [SceneSignal(collection, encoder)])
print(f'{collection.size} segments in the index')

## 3. Pojedyncze zapytanie - top 10 i miary

Wystarczy zmienić `query` na dowolny tekst albo wybrać gotowe zapytanie przez indeks `i`. Poprawny klip to ten, z którego pochodzi opis. Przy własnym tekście ustawić `relevant_vid = None`, żeby pominąć miary.

In [ ]:
queries = load_jsonl(conf.path(cfg['paths']['check']))

# pick a ready query or type your own text
i = 0
query        = queries[i].desc
relevant_vid = queries[i].event_id   # or None for your own text

ranking = pipeline.search(query, k=10)
print('query    :', query)
print('relevant :', relevant_vid, '\n')
for r in ranking:
    mark = '  <== RELEVANT' if r.vid_name == relevant_vid else ''
    print(f'#{r.position:2d}  {r.vid_name}   score={r.score:6.3f}{mark}')

if relevant_vid is not None:
    metrics = query_metrics([r.vid_name for r in ranking], {relevant_vid}, ks)
    print('\nmetrics for this query:', metrics)

## 4. Kontrola poprawności - wszystkie miary

Każdy z dziesięciu opisów klipu jest osobnym zapytaniem, a poprawną odpowiedzią jest klip, z którego opis pochodzi. Raportowane miary: Recall@1, Recall@5, Recall@10 i mAP (dla VATEX równe MRR); MedR i MnR podane diagnostycznie. Przyjęty próg: R@1 w przedziale 40-60% potwierdza poprawność implementacji.

In [ ]:
texts    = [q.desc for q in queries]
relevant = [{q.event_id} for q in queries]

summary = evaluate_matrix(pipeline.scores(texts), collection.vids, relevant, ks)

import pandas as pd

table = pd.Series(summary).to_frame('value')
print('R@1 within the 40-60% band:', 0.40 <= summary['recall@1'] <= 0.60)
table